# **Hugging Face Trainer API: SFT LoRA**

 $\color{red}{\text{Atenção: notebook executado com GPU A100 (T4 no Colab não possui suporte nativo a BF16)}}$

In [ ]:
!pip install transformers datasets accelerate -q


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from datasets import Dataset
import torch


# Um dataset *Supervised Fine-Tuning* (SFT)

In [ ]:
data = [
    {"instruction": "O que é aprendizado de máquina?",
     "response": "Aprendizado de máquina é uma área da inteligência artificial que cria modelos capazes de aprender padrões a partir de dados."},

    {"instruction": "Explique o que é overfitting.",
     "response": "Overfitting ocorre quando um modelo aprende demais os dados de treino e não generaliza bem para novos dados."},

    {"instruction": "O que é uma árvore de decisão?",
     "response": "Uma árvore de decisão é um modelo supervisionado que usa regras em formato de árvore para realizar classificações ou regressões."}
]

dataset = Dataset.from_list(data)
dataset


Dataset({
    features: ['instruction', 'response'],
    num_rows: 3
})

# Load do Modelo e Tokenizer

In [ ]:
model_name = "Qwen/Qwen2.5-0.5B"     # leve e roda no Colab
# model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"  # se você tiver VRAM suficiente
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token  # garantir token de padding

# Carregando o modelo em bfloat16 para usarmos bf16=True na A100
model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.bfloat16)
model.resize_token_embeddings(len(tokenizer))

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Embedding(151665, 896)

# Transformação para o formato de treino

LLMs do tipo causal exigem concatenar `input + resposta` como um único texto.

In [ ]:
def tokenize(example):
    prompt = f"### Pergunta:\n{example['instruction']}\n\n### Resposta:\n{example['response']}"
    tokens = tokenizer(
        prompt,
        truncation=True,
        padding="max_length",
        max_length=256
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_dataset = dataset.map(tokenize, batched=False)
tokenized_dataset


Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Dataset({
    features: ['instruction', 'response', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 3
})

# Configuração do Trainer

In [ ]:
training_args = TrainingArguments(
    output_dir="./finetuned-model",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    num_train_epochs=3,
    logging_steps=1,
    save_steps=20,
    weight_decay=0.01,
    fp16=False,                  # Desative FP16 para evitar o GradScaler
    bf16=True                    # Ative BF16, ideal para GPUs modernas como A100
)

# Treino

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)


## Weights & Biases (`wandb.ai`)

O Hugging Face tem integração automática com plataformas de *experiment tracking* como o Weights & Biases (wandb). Como outros *experiment tracking* ele é uma plataforma para:

- acompanhar métricas de treinamento,
- visualizar loss, learning rate, gradientes,
- salvar checkpoints,
- comparar experimentos,
- registrar hiperparâmetros,
- compartilhar treinamentos

É muito empregado em pesquisa e indústria, **é recomendado**, mas não é obrigatório.

Os scripts da Hugging Face detectam automaticamente se a biblioteca wandb está instalada no ambiente. Para evitar o seu uso:`



In [ ]:
%%script echo off
import os
os.environ["WANDB_DISABLED"] = "true"


off


In [ ]:
trainer.train()


Step,Training Loss
1,14.419188
2,2.244164
3,0.586579


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3, training_loss=5.7499767144521075, metrics={'train_runtime': 5.5689, 'train_samples_per_second': 1.616, 'train_steps_per_second': 0.539, 'total_flos': 4947583500288.0, 'train_loss': 5.7499767144521075, 'epoch': 3.0})

# Chamada do Modelo

In [ ]:
def gerar_texto(pergunta):
    prompt = f"### Pergunta:\n{pergunta}\n\n### Resposta:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(**inputs, max_new_tokens=128)
    print(tokenizer.decode(output[0], skip_special_tokens=True))

gerar_texto("Explique o conceito de regressão linear.")
gerar_texto("Explique o conceito de overfitting.")

### Pergunta:
Explique o conceito de regressão linear.

### Resposta:
A regressão linear é uma técnica de predição que utiliza as variáveis explicativas (variáveis que influenciam a variável dependente) para prever a variável dependente.
### Pergunta:
Explique o conceito de overfitting.

### Resposta:
Overfitting é um problema de aprendizado de máquina em que o modelo é muito bem ajustado a um conjunto de dados, mas não é capaz de prever o comportamento de novos dados.


# LoRA (Low-Rank Adaptation): Fine-tuning Eficiente de LLMs

* **Treina apenas um percentual pequeno (1–2%) dos parâmetros.**

* **Não altera o modelo base (pode salvar apenas os adapters LoRA).**



![imagem](https://github.com/Rogerio-mack/Modelos_de_Linguagem_e_Generativos/blob/main/figures/LoRA.png?raw=true)



O **fine-tuning** completo de Large Language Models (LLMs) pode ser extremamente custoso em termos de recursos computacionais e tempo. Modelos com bilhões de parâmetros exigem GPUs de alta performance e grandes quantidades de memória. Para contornar esse desafio, surgiram técnicas de **Parameter-Efficient Fine-Tuning (PEFT)**, e o **LoRA (Low-Rank Adaptation)** é uma das mais populares e eficazes.

#### A Ideia Central do LoRA

O LoRA baseia-se na observação de que as mudanças nos pesos de um modelo durante o fine-tuning têm uma **rank intrínseco baixo**. Isso significa que as atualizações de peso podem ser representadas por matrizes de rank baixo, que são muito menores do que as matrizes de peso originais do modelo.

Em vez de treinar todos os parâmetros do modelo pré-treinado (W), o LoRA congela os pesos originais do modelo e injeta pares de matrizes de rank baixo (A e B) em cada camada de atenção do Transformer. A matriz original `W` é atualizada por `W + BA`, onde `B` e `A` são matrizes menores, e `r` é o rank intrínseco (geralmente muito menor que as dimensões de `W`).

`W_novo = W_original + B @ A`

Onde:
*   `W_original` é a matriz de pesos pré-treinada (congelada).
*   `B` é uma matriz de `d x r`.
*   `A` é uma matriz de `r x k`.
*   `d` e `k` são as dimensões da matriz `W_original`.
*   `r` é o rank (ex: 4, 8, 16, 32), significativamente menor que `d` e `k`.

Durante o fine-tuning, apenas os pesos das matrizes `A` e `B` são treinados, enquanto `W_original` permanece fixo. Isso reduz drasticamente o número de parâmetros treináveis, exigindo menos memória e computação, e permitindo o fine-tuning em hardware mais modesto.

#### Vantagens do LoRA:

*   **Redução de Parâmetros Treináveis:** Apenas uma pequena fração dos parâmetros é treinada, resultando em menor uso de GPU e treinamento mais rápido.
*   **Menor Custo Computacional:** Menos parâmetros para treinar significa menos FLOPs e menor consumo de energia.
*   **Flexibilidade:** Permite criar múltiplos adaptadores LoRA para diferentes tarefas a partir de um único modelo base, alternando entre eles conforme a necessidade.
*   **Evita Catastrophic Forgetting:** Ao manter os pesos originais congelados, o LoRA ajuda a preservar o conhecimento geral do modelo pré-treinado, evitando a perda de desempenho em tarefas gerais.
*   **Portabilidade:** Os adaptadores LoRA são pequenos e podem ser facilmente compartilhados e implantados.

### QLoRA (Quantized Low-Rank Adaptation): LoRA com Quantização

**QLoRA** é uma extensão do LoRA que leva a eficiência de memória um passo adiante, combinando o LoRA com a **quantização** do modelo base. A quantização é o processo de representar os pesos do modelo com um número menor de bits (por exemplo, de 32-bit float para 4-bit integer), o que reduz significativamente o uso de memória do modelo.

#### A Ideia Central do QLoRA

O QLoRA quantiza o modelo base pré-treinado para 4 bits (ou outro formato de baixa precisão) e, em seguida, aplica o LoRA. Os pesos do modelo base quantizado permanecem congelados e em baixa precisão, enquanto as matrizes `A` e `B` do LoRA são treinadas em 16 bits (BF16 ou FP16). Para o cálculo do gradiente, o QLoRA utiliza uma técnica de **quantização de precisão dupla (Double Quantization)** e uma **quantização de 4 bits com normalização (4-bit NormalFloat - NF4)**, que otimiza a distribuição dos pesos para quantização.

![imagem](https://github.com/Rogerio-mack/Modelos_de_Linguagem_e_Generativos/blob/main/figures/QLoRA.png?raw=true)

Isso significa que o modelo base pode ser carregado em memória com uma fração do tamanho original, e apenas as pequenas matrizes LoRA são atualizadas em maior precisão.

### Quando Usar LoRA vs. QLoRA?

*   **LoRA:** Ideal quando você tem acesso a GPUs com memória suficiente para carregar o modelo base em precisão total (FP16/BF16) e ainda deseja os benefícios de redução de parâmetros treináveis e velocidade.
*   **QLoRA:** A escolha preferida quando a memória da GPU é uma restrição severa, permitindo o fine-tuning de modelos muito maiores que seriam inviáveis com LoRA ou fine-tuning completo.


## Install & import `peft` (`LoRA`)

In [ ]:
!pip install peft -q


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from datasets import Dataset
from peft import LoraConfig, get_peft_model
import torch

## Faz as Mudanças do Modelo Carregado com o `LoRA`

**Atenção**: Se o modelo foi treinado, alterado antes, carregue o modelo novamente.

In [ ]:
model_name = "Qwen/Qwen2.5-0.5B"     # leve e roda no Colab
# model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"  # se você tiver VRAM suficiente

model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.bfloat16)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [ ]:
# Atualiza o torchao para corrigir a incompatibilidade exigida pelo peft
!pip install "torchao>=0.16.0" -q

from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,                       # rank do LoRA
    lora_alpha=32,
    lora_dropout=0.05,
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()   # só para verificar

W1111 13:01:25.798000 464 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184


## Treino

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)


In [ ]:
trainer.train()


Step,Training Loss
1,14.419188
2,14.335985
3,14.277436


TrainOutput(global_step=3, training_loss=14.344202995300293, metrics={'train_runtime': 1.905, 'train_samples_per_second': 4.724, 'train_steps_per_second': 1.575, 'total_flos': 4962531999744.0, 'train_loss': 14.344202995300293, 'epoch': 3.0})

## Chamada do Modelo

In [ ]:
def gerar_texto(pergunta):
    prompt = f"### Pergunta:\n{pergunta}\n\n### Resposta:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(**inputs, max_new_tokens=128)
    print(tokenizer.decode(output[0], skip_special_tokens=True))

gerar_texto("Explique o conceito de regressão linear.")
gerar_texto("Explique o conceito de overfitting.")

### Pergunta:
Explique o conceito de regressão linear.

### Resposta:
A regressão linear é uma técnica de predição utilizada em ciências da computação e de dados para prever valores de uma variável dependente em base de uma variável dependente. A regressão linear é uma técnica de predição que utiliza as linhas de regressão para prever valores de uma variável dependente em base de uma variável dependente. A regressão linear é uma técnica de predição que utiliza as linhas de regressão para prever valores de uma variável dependente em base de uma variável dependente. A regressão linear é uma técnica de predição que utiliza as lin
### Pergunta:
Explique o conceito de overfitting.

### Resposta:
Overfitting é um problema de aprendizado de máquina em que o modelo é muito bem ajustado a um conjunto de dados, mas não é capaz de prever o comportamento de dados que não foram usados para treinar o modelo. Isso ocorre quando o modelo é muito complexo e não tem suficiente informação para prever o c